# 04 · Cell-type composition — GSE216146

Do cell-type proportions differ between chemotherapy (cisplatin) and control (PBS) brains?

Method (propeller-style, Phipson et al. 2022): proportions per **sample** (the unit of replication; 3 vs 3),
arcsin-square-root transform, one linear model per cell type with `composition.formula` (`~ batch + condition`),
Benjamini-Hochberg FDR. With n = 3 per group and 3 residual degrees of freedom, results are **exploratory**.

**Input:** `results/03_annotated.h5ad` · **Output:** tables `results/tables/04_*`, plots `results/plots/04_*`.

Caveats (see `03_annotation`):
- Cell types in `downstream.exclude_cell_types` (low-quality microglia) are removed first.
- Proportions are relative: the very large choroid plexus fraction in D21-2750 (dissection variability) lowers every
  other proportion in that sample. The test is repeated after removing `composition.sensitivity_drop` and renormalizing.
- Neurons/neuroblasts are scarce in two chemotherapy samples; that can be biology or dissection.

In [ ]:
import gc
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import sc_utils as scu

cfg = scu.load_config(ROOT / "config.yaml")
p = cfg["composition"]
scu.setup_plotting()
p

In [ ]:
adata = sc.read_h5ad(cfg["paths"]["results_dir"] / "03_annotated.h5ad")
drop = cfg["downstream"]["exclude_cell_types"]
print("excluded:", adata.obs["cell_type"].isin(drop).sum(), "cells of", drop)
adata = adata[~adata.obs["cell_type"].isin(drop)].copy()
adata.obs["cell_type"] = adata.obs["cell_type"].cat.remove_unused_categories()
obs = adata.obs
adata

## 1. Cells and proportions per sample

In [ ]:
# Per-sample design table (one row per biosample)
meta = (obs[["biosample", "condition", "batch"]].drop_duplicates().set_index("biosample")
        .sort_values(["condition", "batch"]))
meta.index = meta.index.astype(str)
meta["condition"] = meta["condition"].cat.remove_unused_categories()
meta["batch"] = meta["batch"].astype(str)
sample_order = list(meta.index)

counts = pd.crosstab(obs["biosample"].astype(str), obs["cell_type"]).loc[sample_order]
props = counts.div(counts.sum(axis=1), axis=0)
counts.to_csv(cfg["paths"]["tables_dir"] / "04_cell_counts_per_sample.tsv", sep="\t")
props.round(4).to_csv(cfg["paths"]["tables_dir"] / "04_proportions_per_sample.tsv", sep="\t")
meta.join(counts.sum(axis=1).rename("cells"))

In [ ]:
# Stacked bars per sample, with and without choroid plexus
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
colors = dict(zip(adata.obs["cell_type"].cat.categories, adata.uns["cell_type_colors"])) \
    if "cell_type_colors" in adata.uns and len(adata.uns["cell_type_colors"]) == adata.obs["cell_type"].cat.categories.size else None
for ax, (title, tab) in zip(axes, [("all cell types", props),
                                   (f"without {', '.join(p['sensitivity_drop'])}",
                                    counts.drop(columns=p["sensitivity_drop"]).pipe(lambda c: c.div(c.sum(axis=1), axis=0)))]):
    tab.plot(kind="bar", stacked=True, width=0.85, ax=ax, legend=False,
             color=[colors[c] for c in tab.columns] if colors else None)
    ax.set_xticklabels([f"{s}\n{meta.loc[s, 'condition']}" for s in tab.index], rotation=90)
    ax.set_title(title); ax.set_ylabel("fraction of cells"); ax.set_xlabel(""); ax.set_ylim(0, 1); ax.grid(False)
axes[1].legend(title="cell_type", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
fig.tight_layout()
scu.save_fig(fig, cfg, "04_stacked_proportions")
plt.show()

## 2. Composition test (`~ batch + condition`)

In [ ]:
res_all = scu.composition_test(counts, meta, formula=p["formula"])
res_all.round(4)

In [ ]:
# Sensitivity: remove choroid plexus and renormalize the remaining proportions
counts_wo = counts.drop(columns=p["sensitivity_drop"])
res_wo = scu.composition_test(counts_wo, meta, formula=p["formula"])
res_wo.round(4)

In [ ]:
cols = ["log2_ratio", "pval", "fdr"]
tag = "without_" + "_".join(p["sensitivity_drop"]).replace(" ", "_")
both = res_all[cols].add_suffix("_all").join(res_wo[cols].add_suffix(f"_{tag}"), how="left")
both.to_csv(cfg["paths"]["tables_dir"] / "04_composition_test.tsv", sep="\t")
both.round(3)

## 3. Per-sample proportions by condition

Each point is a sample (shape = batch). Shown without choroid plexus, since its size is driven by dissection.

In [ ]:
props_wo = counts_wo.div(counts_wo.sum(axis=1), axis=0)
long = (props_wo.reset_index().melt(id_vars="biosample", var_name="cell_type", value_name="proportion")
        .merge(meta, left_on="biosample", right_index=True))
types = list(res_wo.index)
ncol = 4
nrow = int(np.ceil(len(types) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3 * nrow))
for ax, ct in zip(axes.flat, types):
    d = long[long["cell_type"] == ct]
    sns.boxplot(d, x="condition", y="proportion", ax=ax, color="0.9", showfliers=False, width=0.5)
    sns.stripplot(d, x="condition", y="proportion", hue="batch", ax=ax, size=7, jitter=0.1, legend=ax is axes.flat[0])
    r = res_wo.loc[ct]
    ax.set_title(f"{ct}\nlog2 ratio {r['log2_ratio']:.2f}, p = {r['pval']:.2g}", fontsize=10)
    ax.set_xlabel(""); ax.set_ylabel("proportion"); ax.grid(False)
    ax.tick_params(axis="x", labelrotation=20)
for ax in list(axes.flat)[len(types):]:
    ax.remove()
fig.tight_layout()
scu.save_fig(fig, cfg, "04_proportions_by_condition")
plt.show()

In [ ]:
# UMAP split by condition: cells of one condition coloured, the rest in grey
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
xy = adata.obsm["X_umap"]
for ax, cond in zip(axes, meta["condition"].cat.categories):
    ax.scatter(xy[:, 0], xy[:, 1], s=2, c="lightgrey", rasterized=True)
    sc.pl.umap(adata[adata.obs["condition"] == cond], color="cell_type", ax=ax, show=False, size=6,
               title=f"{cond} ({(adata.obs['condition'] == cond).sum()} cells)",
               legend_loc="right margin" if ax is axes[-1] else None)
fig.tight_layout()
scu.save_fig(fig, cfg, "04_umap_by_condition")
plt.show()

In [ ]:
del adata
_ = gc.collect()